# From 2D Satellite Image to 3D Surface

## Building a Mini DSM Pipeline in Python

An optical satellite image is fundamentally two-dimensional. A **Digital Surface Model (DSM)** adds surface height so that every horizontal position \((x,y)\) is associated with an elevation \(z\).

This project builds an educational, reproducible Python pipeline:

\[
\text{optical image}
\rightarrow
\text{preprocessing}
\rightarrow
\text{DSM model}
\rightarrow
\text{predicted surface}
\rightarrow
\text{reference comparison}
\]

The goals are to:

1. understand the geospatial and machine-learning steps required to turn a 2D image into a height raster;
2. design the processing logic so that the same Python functions can later power a small interactive app.

> **Project status**  
> This first page establishes the data model, processing pipeline, validation logic and visualization workflow. The deep-learning inference step is intentionally kept as a clearly defined interface until a suitable single-image DSM model and compatible imagery are selected.

**→ Try the interactive Mini DSM app** *(link will be added after deployment)*


## What are DEM, DSM and DTM?

| Product | Meaning | What the height represents |
|---|---|---|
| **DEM** | Digital Elevation Model | General raster representation of elevation |
| **DSM** | Digital Surface Model | Top visible surface: ground, roofs, trees and other objects |
| **DTM** | Digital Terrain Model | Bare-earth terrain after buildings and vegetation are removed |

All can be represented conceptually as

\[
z=f(x,y).
\]

If DSM and DTM use the same grid, object height can approximately be estimated as

\[
h_{\mathrm{object}}=z_{\mathrm{DSM}}-z_{\mathrm{DTM}}.
\]

For this project the target is a **surface model**, so buildings and vegetation are part of the geometry rather than noise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 100, 240)
terrain = 8 + 0.018 * x + 0.7 * np.sin(x / 17)

surface = terrain.copy()
surface[(x > 24) & (x < 43)] += 8.5
surface[(x > 62) & (x < 69)] += 5.5
surface[(x > 70) & (x < 77)] += 7.5

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, surface, linewidth=2.2, label="DSM: visible surface")
ax.plot(x, terrain, linewidth=2.0, label="DTM: bare terrain")
ax.fill_between(x, terrain, surface, alpha=0.15)
ax.set_xlabel("Horizontal distance")
ax.set_ylabel("Elevation")
ax.set_title("DSM keeps objects above the terrain")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## 1. Pipeline architecture

```text
Optical satellite image
        │
        ▼
1. Read raster + metadata
        │
        ▼
2. Reproject / crop / resample
        │
        ▼
3. Build model-ready image tensor
        │
        ▼
4. Single-image DSM inference
        │
        ▼
5. Export predicted GeoTIFF
        │
        ├───────────────┐
        ▼               ▼
Reference DSM/DEM    Visualization
        │               │
        └──────┬────────┘
               ▼
        MAE · RMSE · Bias
               │
               ▼
        Error map + 3D surface
```

The scientific processing should not depend on the interface. The notebook explains the pipeline; the later Streamlit app will call the same functions.


## 2. Environment and reproducibility

Core tools:

- **NumPy / pandas** — array and tabular processing
- **Rasterio** — raster I/O, CRS, transforms and reprojection
- **GeoPandas** — vector geometry and study-area metadata
- **Matplotlib / Plotly** — static and interactive visualization
- **PyTorch or TensorFlow** — model inference once a DSM model is selected


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "geo" / "dsm_pipeline_data"
OUTPUT_DIR = ROOT / "geo" / "dsm_pipeline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)


## 3. Choose a compact study area

A useful DSM test area should contain several surface types:

- buildings with clear roof geometry;
- trees or vegetation;
- open ground;
- roads or other relatively flat surfaces.

A small urban subset around **Uppsala** is a practical first test.

> **Resolution warning**  
> Sentinel-2 is useful for learning the raster-processing workflow, but its common visible/NIR bands have 10 m pixels. That is too coarse to resolve individual roofs with LiDAR-like detail. A genuine high-resolution single-image DSM experiment therefore needs imagery and a model whose spatial resolution are compatible with the desired surface detail.


In [ ]:
study_area = {
    "name": "Uppsala test area",
    "crs": "EPSG:4326",
    "bbox": [17.61, 59.84, 17.68, 59.88],  # west, south, east, north
}

pd.Series(study_area, name="value").to_frame()


## 4. Input: one optical satellite image

For every raster we need both pixel values and spatial metadata:

- CRS;
- affine transform;
- pixel size;
- spatial bounds;
- band order;
- nodata values.

A robust loader should therefore return **both the image array and its spatial metadata**.


In [ ]:
def describe_raster(array, *, crs=None, nodata=None):
    array = np.asarray(array)

    if array.ndim == 2:
        bands, height, width = 1, *array.shape
    elif array.ndim == 3:
        bands, height, width = array.shape
    else:
        raise ValueError("Expected a 2D raster or a band-first 3D raster.")

    finite = np.isfinite(array)

    return pd.Series({
        "bands": bands,
        "height_px": height,
        "width_px": width,
        "crs": crs,
        "nodata": nodata,
        "finite_fraction": float(finite.mean()),
        "min": float(np.nanmin(array)),
        "max": float(np.nanmax(array)),
    })

demo_rgb = np.stack([
    np.linspace(0, 1, 100).reshape(10, 10),
    np.flipud(np.linspace(0, 1, 100).reshape(10, 10)),
    np.full((10, 10), 0.4),
])

describe_raster(demo_rgb, crs="demo")


## 5. Preprocessing: make the image model-ready

Common preprocessing operations include:

1. selecting the expected bands;
2. masking invalid pixels;
3. cropping to the study area;
4. resampling to the expected resolution;
5. converting data types;
6. normalizing pixel values;
7. tiling a large raster into model-sized patches.

> **Never lose the georeferencing when converting the raster into a model tensor.**


In [ ]:
def percentile_normalize(image, lower=2, upper=98):
    image = np.asarray(image, dtype="float32")
    out = np.empty_like(image)

    for band in range(image.shape[0]):
        values = image[band]
        lo, hi = np.nanpercentile(values, [lower, upper])

        if hi <= lo:
            out[band] = 0
            continue

        clipped = np.clip(values, lo, hi)
        out[band] = (clipped - lo) / (hi - lo)

    return out

demo_normalized = percentile_normalize(demo_rgb)
print("Normalized range:", demo_normalized.min(), "to", demo_normalized.max())


## 6. DSM inference: keep the model behind a clean interface

The model should be replaceable without rewriting the rest of the pipeline.

```python
predicted_dsm = predict_dsm(model_input)
```

The function should return a **2D height raster** in physical elevation units.

This notebook does **not** invent a DSM from RGB values. A true single-image height estimate requires a trained model with known preprocessing, output conventions and compatible imagery.


In [ ]:
def predict_dsm(model_input, model=None):
    # Future interface for a validated single-image DSM model.
    if model is None:
        raise NotImplementedError(
            "DSM inference is not connected yet. "
            "Select a trained single-image DSM model first."
        )

    raise NotImplementedError


## 7. Reference elevation and grid alignment

Before calculating errors, predicted and reference rasters must have compatible:

- CRS;
- spatial extent;
- pixel size;
- grid origin;
- array shape;
- vertical datum / elevation convention.

This is a **geospatial registration problem**, not merely subtraction of two matrices.


## 8. Validation

For valid pixels,

\[
e_i=z_i^{\mathrm{pred}}-z_i^{\mathrm{ref}}.
\]

### Mean Absolute Error

\[
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|e_i|
\]

### Root Mean Square Error

\[
\mathrm{RMSE}
=
\sqrt{\frac{1}{n}\sum_{i=1}^{n}e_i^2}
\]

### Bias

\[
\mathrm{Bias}=\frac{1}{n}\sum_{i=1}^{n}e_i
\]

The error map is equally important because it reveals **where** the model fails.


In [ ]:
def elevation_metrics(predicted, reference, valid_mask=None):
    predicted = np.asarray(predicted, dtype="float64")
    reference = np.asarray(reference, dtype="float64")

    if predicted.shape != reference.shape:
        raise ValueError("Predicted and reference rasters must have the same shape.")

    valid = np.isfinite(predicted) & np.isfinite(reference)

    if valid_mask is not None:
        valid &= np.asarray(valid_mask, dtype=bool)

    if not valid.any():
        raise ValueError("No valid pixels are available for comparison.")

    error = predicted[valid] - reference[valid]

    return pd.Series({
        "n_pixels": int(valid.sum()),
        "MAE_m": np.mean(np.abs(error)),
        "RMSE_m": np.sqrt(np.mean(error**2)),
        "Bias_m": np.mean(error),
    })

reference_demo = np.array([[10.0, 11.0], [12.0, 13.0]])
prediction_demo = np.array([[10.4, 10.7], [12.6, 12.8]])

elevation_metrics(prediction_demo, reference_demo)


## 9. Spatial error map

\[
\Delta z(x,y)=z_{\mathrm{pred}}(x,y)-z_{\mathrm{ref}}(x,y)
\]

- positive error → prediction too high;
- negative error → prediction too low;
- near zero → close agreement.


In [ ]:
yy, xx = np.mgrid[-2:2:160j, -2:2:180j]

reference_surface = (
    14
    + 1.5 * xx
    + 0.7 * np.sin(2 * yy)
    + 8 * ((np.abs(xx + 0.6) < 0.35) & (np.abs(yy - 0.2) < 0.45))
)

predicted_surface = (
    reference_surface
    + 0.35 * np.sin(3 * xx)
    - 0.25 * np.cos(2 * yy)
)

error_surface = predicted_surface - reference_surface

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(error_surface, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Example spatial elevation error: predicted − reference")
ax.set_axis_off()
fig.colorbar(im, ax=ax, label="Elevation error")
plt.show()


## 10. 3D surface visualization

A 3D view makes a DSM intuitive. The final page can use Plotly so the reader can rotate, zoom and inspect elevations on hover.

The 3D view is a **communication tool**, not the validation itself.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

step = 4
ax.plot_surface(
    xx[::step, ::step],
    yy[::step, ::step],
    predicted_surface[::step, ::step],
    linewidth=0,
    antialiased=True,
    alpha=0.95,
)

ax.set_title("Example DSM-style height surface")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("Elevation")
ax.view_init(elev=33, azim=-125)
plt.show()


## 11. Data-quality checklist

| Check | Why it matters |
|---|---|
| Is the optical image georeferenced? | The output must return to a real-world location |
| Are bands in the order expected by the model? | Wrong band order can invalidate inference |
| Is the spatial resolution compatible with training data? | Models may not generalize across scales |
| Are predicted and reference rasters on the same grid? | Pixel metrics otherwise compare different places |
| Are nodata pixels excluded? | Missing values must not become elevation errors |
| Are vertical units/datums compatible? | Datum offsets can look like model bias |
| Was resampling applied? | Resampling changes detail |
| Is the reference a DSM rather than a DTM? | Buildings/trees should not be judged against bare earth |

The aim is to make the result **auditable**, not merely attractive.


## 12. Static Data Lab page vs interactive app

This page is static HTML:

```text
Notebook
   ↓ run Python
Saved outputs
   ↓ build
Static Data Lab page
```

The later Python app will be interactive:

```text
User selects image / study area
          ↓
      Streamlit
          ↓
Python DSM pipeline
          ↓
prediction + metrics + maps
```

Both should use the **same processing functions**.

A first app can expose only a few reliable controls:

- choose a prepared study area;
- choose one optical scene;
- click **Generate DSM**;
- compare **Input / Predicted DSM / Reference / Error**;
- inspect MAE, RMSE and bias;
- rotate a 3D surface.


## 13. What this project demonstrates

- optical Earth-observation data;
- raster metadata and coordinate systems;
- model-ready preprocessing;
- a replaceable ML inference interface;
- georeferenced output;
- reference-data alignment;
- quantitative validation;
- spatial error analysis;
- 2D and 3D visualization;
- reproducible Python code.

The project is deliberately built as a **pipeline**, making the transition from research experiment to user-facing prototype much easier.

### Next milestones

1. Select a real optical dataset and compact study area.
2. Select a compatible single-image DSM model.
3. Obtain a trustworthy reference DSM / LiDAR product.
4. Implement raster alignment and tiling.
5. Run the first real inference.
6. Add Plotly comparison views.
7. Refactor reusable code into a small Python module.
8. Build and deploy the Streamlit demo.
9. Replace the placeholder app link at the top of this page.


## References and data sources to connect

The final implementation should document exact versions and licenses. Likely sources include:

- Copernicus Sentinel-2 for optical Earth-observation imagery and workflow experiments;
- Copernicus DEM where its resolution is appropriate;
- Swedish elevation/LiDAR data where access and licensing permit higher-resolution validation;
- the documentation and publication associated with the selected single-image DSM model.

Official sources should be linked directly once the real datasets are connected.
